In [14]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from tqdm.notebook import tqdm

In [15]:
# Directories
PROJECT = "C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"
DATA = os.path.join(PROJECT, "Data")

# Input files
INPUT_DATA = os.path.join(DATA, "merged_master.pkl")

# Model prediction files
# Key = column name in the dataframe, Value = prediction filename
MODELS = {
    'lr_2': 'predictions_linear_regression_input=2.pkl',
    'lr_56': 'predictions_linear_regression_input=56.pkl',
}

# Time range to run regressions
START_DATE = '2012-01-01'
END_DATE = '2022-12-31'

# Prediction target
TARGET = 'f_cumret1'

# Put together all predictions

In [16]:
# Load the original aggregated tweets data
df = pd.read_pickle(INPUT_DATA)[['date', 'permno', 'ticker', TARGET, 'log_volume']].copy()
df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)]

for model_col, filename in MODELS.items():
    # Load model predictions
    model_predictions = pd.read_pickle(os.path.join(DATA, filename)).drop(columns=['index', 'ticker'])
    model_predictions.columns = ['date', 'permno', model_col]
    
    # Merge with master data
    df = pd.merge(df, model_predictions, on=['date', 'permno'])  # Clean merge

In [17]:
# Cross-sectionally de-mean the target and predictions each day
daily_means = df.groupby('date')[[TARGET] + list(MODELS.keys())].transform('mean')
df[TARGET] = df[TARGET] - daily_means[TARGET]
for col in list(MODELS.keys()):
    df[col] = df[col] - daily_means[col]

print("De-meaned target and predictions by date")
print(f"  Mean of {TARGET} after de-meaning: {df[TARGET].mean():.2e}")
for col in list(MODELS.keys()):
    print(f"  Mean of {col} after de-meaning: {df[col].mean():.2e}")

De-meaned target and predictions by date
  Mean of f_cumret1 after de-meaning: 1.33e-20
  Mean of lr_2 after de-meaning: 4.42e-22
  Mean of lr_56 after de-meaning: 4.42e-22


# Run sentiment regressions

In [20]:
reg_results = []
for pred_col in tqdm(MODELS.keys(), desc="Running regressions"):
    reg_data = df[['permno','date', TARGET, pred_col]].dropna().copy()
    reg_data['date'] = pd.to_datetime(reg_data['date'])
    reg_data['date'] = reg_data['date'].dt.year * 10000 + reg_data['date'].dt.month*100 + reg_data['date'].dt.day 
    reg_data = reg_data.rename(columns={pred_col: 'pred'})
    y = reg_data[TARGET]
    # X = sm.add_constant(reg_data['pred'])
    X = reg_data['pred']
    model = sm.OLS(y, X, missing='drop').fit(cov_type='cluster',cov_kwds={'groups':np.array(reg_data[['permno','date']])})
    reg_results.append(model)

Running regressions:   0%|          | 0/2 [00:00<?, ?it/s]

# Render the results in a table

In [21]:
from latex_table import linear_regression

# Format and save the table
vars_to_include = ['pred', 'const']
var_names = ["Prediction", "Const."]
rename_dict = dict(zip(vars_to_include, var_names))

tbl = linear_regression(reg_results)
tbl.rename_variables(rename_dict)
tbl.columns = MODELS.keys()
tbl.obs = True
tbl.R2 = True
tbl.float_format = ".6f"
tbl.render(midrule=True)
tbl.tbl

lr_2              lr_56
Prediction    0.799930THREESTAR  0.332249THREESTAR
                     (0.140218)         (0.073368)
                                    ADDMIDRULEHERE
N                    12,047,638         12,047,638